# Train ACT on `salik/dex-arm-01-sim`

Run this on a **free Colab T4**. It trains an ACT policy on the 37 simulated
grasp demonstrations of DEX ARM 01 and pushes the checkpoint to the Hub.

**What this proves:** the dataset loads with `LeRobotDataset`, the training path
runs end to end, loss decreases, and a checkpoint is published.

**What this does not prove:** that the hand can grasp. The data is 37 successful
closures of one cylinder class in one simulator. Do not report a success rate
from this run as hardware or even sim performance.

Author: generated for the DEX ARM 01 repository. Commands follow `lerobot`
0.6.1 (`README.md` quickstart and `configs/train.py` field names).

## 1. Check the runtime

Runtime → Change runtime type → **T4 GPU**. `lerobot` 0.6.1 requires Python ≥ 3.12.
If the version below is older than 3.12, pick a newer Colab runtime image.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys
print('python', sys.version.split()[0])
assert sys.version_info >= (3, 12), 'lerobot 0.6.1 needs Python 3.12+'

## 2. Install LeRobot

Pinned so a rerun in a month does not silently change behaviour. Takes a few
minutes on a fresh Colab VM (torch is the slow part).

In [ ]:
!pip install -q "lerobot==0.6.1"
!lerobot-train --help | head -20

## 3. Log in to the Hub

Use a **write** token. This is needed to read the dataset and to push the
checkpoint. Colab keeps it only for the session.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Load the dataset and check the contract

These assertions are the point of the cell: if the dataset stops matching what
the notebook expects, it fails here instead of half-way through training.

In [ ]:
from huggingface_hub import HfApi
from lerobot.datasets.lerobot_dataset import LeRobotDataset

# LeRobot refuses a dataset whose repo is not tagged with its codebase version.
# The published repo already carries `v3.0`; this keeps a fork working.
try:
    HfApi().create_tag('salik/dex-arm-01-sim', tag='v3.0', repo_type='dataset')
except Exception as exc:  # tag already exists
    print('tag check:', type(exc).__name__)

ds = LeRobotDataset("salik/dex-arm-01-sim")
print('episodes', ds.num_episodes, 'frames', ds.num_frames)
print('fps', ds.fps, 'tasks', ds.meta.total_tasks)
print('features', sorted(ds.meta.features))

sample = ds[0]
assert sample['observation.state'].shape[-1] == 27
assert sample['observation.environment_state'].shape[-1] == 7
assert sample['action'].shape[-1] == 27
assert ds.num_episodes == 37
print('contract OK:', sample['observation.state'].shape, sample['action'].shape)

## 5. Train

ACT reads `observation.state` and `observation.environment_state`, and predicts
27-DoF action chunks. It needs no camera because the dataset carries the object
pose — an ACT requirement that a state-only dataset would fail.

20 000 steps at batch 16 on 1722 frames is a pipeline demonstration, not a
converged policy; expect visible overfitting well before the end. Watch that
the loss falls in the first few hundred steps, then that the checkpoint lands
on the Hub. `--save_freq=5000` gives four intermediate checkpoints.

In [ ]:
!lerobot-train \
  --policy.type=act \
  --dataset.repo_id=salik/dex-arm-01-sim \
  --output_dir=outputs/act_dex_arm \
  --job_name=dex_arm_act \
  --steps=20000 \
  --batch_size=16 \
  --save_freq=5000 \
  --num_workers=2 \
  --wandb.enable=false \
  --policy.push_to_hub=true \
  --policy.repo_id=salik/dex-arm-01-act

## 6. Confirm the checkpoint

Read the published model back from the Hub rather than trusting the upload
message.

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files('salik/dex-arm-01-act')
print(len(files), 'files on the Hub')
for f in files[:20]:
    print(' ', f)
assert any(f.endswith('config.json') for f in files), 'no config.json: the checkpoint did not upload'

## 7. Record what you got

For `RESULTS.md`, capture: GPU name, wall-clock training time, final loss, the
checkpoint repo, and the fact that the data is simulated. Anything a reader
cannot reproduce from the dataset and this notebook does not belong in the
results table.

**Do not** evaluate this checkpoint against the 37 training episodes and call
the number a success rate — it was trained on them.